In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import networkx as nx
import zipfile, re
#project imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils
import plotting
plotting.set_plot_style()
print(f"Figures will be saved to: {cfg.FIGURE_DIR}")

Figures will be saved to: /Users/sargun/Desktop/DAMB-DAv3c/results/figures


In [12]:
# =========================================================
# Input Paths
# =========================================================
PATH_BASELINE = (
    cfg.DATA_DIR / "metadata" / "baseline.csv"
)

INST_ROOT = (
    cfg.RAW_DIR / "mqlib_instances"
)

# =========================================================
# Output Paths
# =========================================================

OUTPUT_PATH = (
    cfg.PROCESSED_DIR / "metrics.csv"
)


In [13]:
baseline = pd.read_csv(PATH_BASELINE)

In [14]:
baseline.head()

,name,nodes,edges,density,limit,has_leafs,int_only
0,G55,4969,12498,0.001013,13.945,True,True
1,G56,4969,12498,0.001013,13.922,True,True
2,G57,5000,10000,0.000800,11.299,False,True
3,G58,5000,29570,0.002366,11.876,False,True
4,G59,5000,29570,0.002366,15.875,False,True


In [15]:
# reading edges
def read_edgelist_from_zip(zip_path: Path):
    """
    Reads the first file in the zip as (u v [w]) edgelist. Returns (n, edges, inner_name).
    """
    with zipfile.ZipFile(zip_path, "r") as zfp:
        inner_files = [n for n in zfp.namelist() if not n.endswith("/")]
        if not inner_files:
            raise ValueError(f"No files inside zip: {zip_path}")
        inner_name = inner_files[0]
        data = zfp.read(inner_name).decode("utf-8", errors="ignore").splitlines()

    edges = []
    nodes = set()

    for line in data:
        line = line.strip()
        if not line or line.startswith(("#", "%", "c")):
            continue
        if line.startswith("p "):  # skip DIMACS-like headers
            continue

        parts = re.split(r"\s+", line)
        if len(parts) < 2:
            continue
        if len(parts) == 2:
            continue

        try:
            u = int(parts[0])
            v = int(parts[1])
            w = float(parts[2]) if len(parts) >= 3 else 1.0
            
        except ValueError:
            continue

        edges.append((u, v, w))
        nodes.update([u, v])

    if not edges:
        raise ValueError(f"No edges parsed from {zip_path} (inner: {inner_name})")

    edges_used = len(edges)
    # Normalize to 0-based indexing (many formats are 1-based)
    has_zero = any(u == 0 or v == 0 for u, v, _ in edges)
    if not has_zero:
        edges = [(u - 1, v - 1, w) for u, v, w in edges]
        nodes = {x - 1 for x in nodes}

    n = max(nodes) + 1
    n_used = len(nodes)
    return n, edges, inner_name, n_used, edges_used


In [ ]:
total = len(baseline )
unique_weights = []
weight = []
for i, (_, x) in enumerate(baseline.iterrows(), 1):
    graph_name = x['name']
    progress = (i / total) * 100
    sys.stdout.write(
        f"\rProcessed {i}/{total} "
        f"({progress:.2f}%) | Current: {graph_name}"
    )
    sys.stdout.flush()

    zip_path = utils.find_zip_for_graph(graph_name)
    
    if zip_path is None:
        print(f'Missing ZIP for {graph_name}')
        continue
    
    n, edges, inner, n_used, edges_used = read_edgelist_from_zip(zip_path)
    weights = []

    for x in edges:
        weights.append(x[2])

    unique_weights.append(len(set(weights)))
    weight.append(weights)

Processed 363/3506 (10.35%) | Current: g000184n99

In [5]:
baseline["unique weights"] = unique_weights
baseline["weights"] = weight
baseline.to_csv(OUTPUT_PATH,index=False)